# RobinReal — Prototype + Debug Dashboard

**Interface:** `search(query: str) -> SearchResult` carrying the ranked listings *and* a per-stage debug trace.

```
query ──► extract_hard_facts ──► hard filter ──► extract_soft_facts ──► filter_soft_facts ──► rank_listings ──► results
             │                      │                  │                       │                    │
             └── trace ─────────────┴── trace ─────────┴── trace ──────────────┴── trace ───────────┴── trace ──► dashboard
```

The four participant functions (`extract_hard_facts`, `extract_soft_facts`, `filter_soft_facts`, `rank_listings`) **match the harness signatures** in [`challenge harness/app/participant/`](../../challenge%20harness/app/participant/) 1:1. Develop here, then copy the function body into the matching harness file — no interface translation.

## 1. Setup

In [ ]:
%pip install -q folium pandas

## 2. Schemas

Plain dataclasses mirroring `challenge harness/app/models/schemas.py`. When copying code into the harness, swap these for the pydantic imports.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Literal

@dataclass
class HardFilters:
    city: list[str] | None = None
    postal_code: list[str] | None = None
    canton: str | None = None
    min_price: int | None = None
    max_price: int | None = None
    min_rooms: float | None = None
    max_rooms: float | None = None
    latitude: float | None = None
    longitude: float | None = None
    radius_km: float | None = None
    features: list[str] | None = None
    offer_type: str | None = None
    object_category: list[str] | None = None
    limit: int = 25
    offset: int = 0
    sort_by: Literal['price_asc', 'price_desc', 'rooms_asc', 'rooms_desc'] | None = None

@dataclass
class ListingData:
    id: str
    title: str
    description: str | None = None
    street: str | None = None
    city: str | None = None
    postal_code: str | None = None
    canton: str | None = None
    latitude: float | None = None
    longitude: float | None = None
    price_chf: int | None = None
    rooms: float | None = None
    living_area_sqm: int | None = None
    available_from: str | None = None
    image_urls: list[str] | None = None
    hero_image_url: str | None = None
    original_listing_url: str | None = None
    features: list[str] = field(default_factory=list)
    offer_type: str | None = None
    object_category: str | None = None
    object_type: str | None = None

@dataclass
class RankedListingResult:
    listing_id: str
    score: float
    reason: str
    listing: ListingData

# Notebook-only extras for debug dashboard.
@dataclass
class StageTrace:
    name: str
    ms: float
    info: dict[str, Any] = field(default_factory=dict)

@dataclass
class SearchResult:
    query: str
    hard_filters: HardFilters
    soft_facts: dict[str, Any]
    candidates: list[dict[str, Any]]        # after hard filter, before soft filter
    soft_filtered: list[dict[str, Any]]     # after soft filter
    ranked: list[RankedListingResult]       # final output (matches harness ListingsResponse.listings)
    traces: list[StageTrace] = field(default_factory=list)

## 3. Pseudo pipeline (stubs)

The four `extract_*`, `filter_soft_facts`, `rank_listings` functions below are the **only** pieces teammates need to implement. Signatures match harness exactly — replace the bodies with real logic.

In [ ]:
import time, re

# ---------------------------------------------------------------------------
# In-memory listing pool (dicts, matching the row shape the harness produces
# from SQLite: lower_snake_case keys, listing_id + rooms + price + latitude...).
# Replace with real DB access later — the signatures below won't change.
# ---------------------------------------------------------------------------
PSEUDO_POOL: list[dict[str, Any]] = [
    {'listing_id': '1', 'title': 'Helle 3.5-Zi Wengistrasse',  'city': 'Zürich',     'price': 2650, 'rooms': 3.5, 'latitude': 47.374, 'longitude': 8.530, 'description': 'Bright Minergie apartment, south-facing, balcony.',    'features': ['balcony']},
    {'listing_id': '2', 'title': 'Moderne Wohnung Oerlikon',   'city': 'Zürich',     'price': 2450, 'rooms': 3.5, 'latitude': 47.411, 'longitude': 8.544, 'description': 'Modern maisonette near ETH, roof terrace, quiet.',     'features': ['balcony', 'parking']},
    {'listing_id': '3', 'title': 'Gemütliche Wohnung Seebach', 'city': 'Zürich',     'price': 2200, 'rooms': 3.5, 'latitude': 47.423, 'longitude': 8.547, 'description': 'Cozy quiet flat, near Seebach station, parking.',     'features': ['balcony', 'parking']},
    {'listing_id': '4', 'title': 'Studio Genève Plainpalais',  'city': 'Genève',     'price': 1800, 'rooms': 1.0, 'latitude': 46.195, 'longitude': 6.141, 'description': 'Modern studio, quiet neighborhood, rooftop views.', 'features': []},
    {'listing_id': '5', 'title': '3-Zi Winterthur Altstadt',   'city': 'Winterthur', 'price': 1750, 'rooms': 3.0, 'latitude': 47.500, 'longitude': 8.724, 'description': 'Family-friendly flat, bright rooms, parking.',      'features': ['parking']},
    {'listing_id': '6', 'title': '4-Zi Winterthur Töss',       'city': 'Winterthur', 'price': 2400, 'rooms': 4.0, 'latitude': 47.478, 'longitude': 8.716, 'description': 'Spacious, near schools and park, south-facing balcony.', 'features': ['balcony']},
]

def _timed(fn, *args, **kw):
    t0 = time.perf_counter(); out = fn(*args, **kw); return out, (time.perf_counter() - t0) * 1000


# ===========================================================================
# PARTICIPANT FUNCTION 1 — hard_fact_extraction.py
# Signature matches harness exactly.
# ===========================================================================
CITY_TOKENS = {'zurich': 'Zürich', 'zürich': 'Zürich', 'geneva': 'Genève', 'winterthur': 'Winterthur', 'basel': 'Basel'}
KNOWN_FEATURES = {'balcony', 'parking', 'elevator', 'garden'}

def extract_hard_facts(query: str) -> HardFilters:
    q = query.lower()
    hf = HardFilters()
    for tok, city in CITY_TOKENS.items():
        if tok in q:
            hf.city = [city]; break
    m = re.search(r'under\s*(?:chf)?\s*(\d{3,5})', q)
    if m: hf.max_price = int(m.group(1))
    m = re.search(r'(\d(?:\.\d)?)\s*[- ]?room', q)
    if m:
        rooms = float(m.group(1))
        hf.min_rooms = hf.max_rooms = rooms
    feats = [f for f in KNOWN_FEATURES if f in q]
    if feats: hf.features = feats
    return hf


# ===========================================================================
# HARNESS-PROVIDED: hard filter SQL (app/core/hard_filters.py).
# In the notebook we reimplement against PSEUDO_POOL. The real harness
# handles this — your team does not write this function.
# ===========================================================================
def _hard_filter_pool(hf: HardFilters) -> list[dict[str, Any]]:
    out = PSEUDO_POOL
    if hf.city:       out = [r for r in out if r.get('city') in hf.city]
    if hf.min_price is not None: out = [r for r in out if r.get('price', 0) >= hf.min_price]
    if hf.max_price is not None: out = [r for r in out if r.get('price', 0) <= hf.max_price]
    if hf.min_rooms is not None: out = [r for r in out if r.get('rooms', 0) >= hf.min_rooms]
    if hf.max_rooms is not None: out = [r for r in out if r.get('rooms', 0) <= hf.max_rooms]
    if hf.features:
        for feat in hf.features:
            out = [r for r in out if feat in (r.get('features') or [])]
    return out[hf.offset : hf.offset + hf.limit]


# ===========================================================================
# PARTICIPANT FUNCTION 2 — soft_fact_extraction.py
# Signature matches harness: returns a free-form dict of soft signals.
# ===========================================================================
SOFT_KEYWORDS = ['bright', 'quiet', 'modern', 'family', 'central', 'view', 'cozy']

def extract_soft_facts(query: str) -> dict[str, Any]:
    q = query.lower()
    prefs = {kw: 1.0 for kw in SOFT_KEYWORDS if kw in q}
    if any(p in q for p in ['not too expensive', 'affordable', 'cheap']):
        prefs['affordable'] = 1.0
    return {'raw_query': query, 'preferences': prefs}


# ===========================================================================
# PARTICIPANT FUNCTION 3 — soft_filtering.py
# Signature matches harness: trims / drops candidates based on soft signals.
# Default passthrough; teams can replace with stricter logic.
# ===========================================================================
def filter_soft_facts(candidates: list[dict[str, Any]], soft_facts: dict[str, Any]) -> list[dict[str, Any]]:
    return candidates


# ===========================================================================
# PARTICIPANT FUNCTION 4 — ranking.py
# Signature matches harness: scores + reshapes into RankedListingResult list.
# ===========================================================================
def rank_listings(candidates: list[dict[str, Any]], soft_facts: dict[str, Any]) -> list[RankedListingResult]:
    prefs: dict[str, float] = soft_facts.get('preferences', {})
    ranked: list[tuple[float, str, dict[str, Any]]] = []
    for c in candidates:
        desc = (c.get('description') or '').lower()
        hits = [kw for kw in prefs if kw != 'affordable' and kw in desc]
        score = 0.5 + 0.1 * len(hits)
        if 'affordable' in prefs:
            mean_price = sum(x.get('price', 0) for x in candidates) / max(1, len(candidates))
            if c.get('price', 0) < mean_price:
                score += 0.1; hits.append('affordable')
        reason = ('matches ' + ', '.join(hits)) if hits else 'Matched hard filters; no soft signals found.'
        ranked.append((score, reason, c))
    ranked.sort(key=lambda x: x[0], reverse=True)
    return [
        RankedListingResult(
            listing_id=str(c['listing_id']),
            score=round(score, 2),
            reason=reason,
            listing=ListingData(
                id=str(c['listing_id']),
                title=c.get('title', ''),
                description=c.get('description'),
                city=c.get('city'),
                latitude=c.get('latitude'),
                longitude=c.get('longitude'),
                price_chf=c.get('price'),
                rooms=c.get('rooms'),
                features=c.get('features') or [],
            ),
        )
        for score, reason, c in ranked
    ]

## 4. `search()` — wires stages + records trace

In [ ]:
def search(query: str) -> SearchResult:
    """Mirror of the harness search_service: runs the five pipeline stages and records timings."""
    traces: list[StageTrace] = []

    hf, ms = _timed(extract_hard_facts, query)
    traces.append(StageTrace('extract_hard_facts', ms, {'filters': {k: v for k, v in hf.__dict__.items() if v not in (None, [], 25, 0)}}))

    candidates, ms = _timed(_hard_filter_pool, hf)
    traces.append(StageTrace('hard_filter', ms, {'pool_size': len(PSEUDO_POOL), 'candidates': len(candidates)}))

    sf, ms = _timed(extract_soft_facts, query)
    traces.append(StageTrace('extract_soft_facts', ms, {'preferences': sf.get('preferences', {})}))

    soft_filtered, ms = _timed(filter_soft_facts, candidates, sf)
    traces.append(StageTrace('filter_soft_facts', ms, {'survivors': len(soft_filtered)}))

    ranked, ms = _timed(rank_listings, soft_filtered, sf)
    traces.append(StageTrace('rank_listings', ms, {'ranked': len(ranked)}))

    return SearchResult(
        query=query, hard_filters=hf, soft_facts=sf,
        candidates=candidates, soft_filtered=soft_filtered, ranked=ranked,
        traces=traces,
    )

## 5. Map view — price pins

Airbnb-style price pills rendered at each listing's lat/lon. Click a pin for title, rooms, score, and reason.

In [ ]:
import folium
from folium.features import DivIcon

PIN_CSS = (
    "display:inline-block;padding:4px 10px;border-radius:16px;"
    "background:#fff;border:1.5px solid #222;box-shadow:0 1px 4px rgba(0,0,0,.25);"
    "font:600 12px/1 -apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif;"
    "color:#222;white-space:nowrap;"
)
PIN_CSS_TOP = PIN_CSS.replace('#fff', '#111').replace('color:#222', 'color:#fff').replace('#222', '#111')

def _popup_html(r: RankedListingResult, rank: int) -> str:
    l = r.listing
    img = f'<img src="{l.hero_image_url}" style="width:100%;border-radius:6px;margin-bottom:6px">' if l.hero_image_url else ''
    price = f'CHF {l.price_chf:,.0f}/mo' if l.price_chf else 'price n/a'
    rooms = f'{l.rooms:g} rooms' if l.rooms else ''
    return (
        f'<div style="font:13px/1.4 -apple-system,BlinkMacSystemFont,sans-serif;width:220px">'
        f'{img}'
        f'<div style="font-weight:700;margin-bottom:2px">#{rank} · {l.title}</div>'
        f'<div style="color:#555">{l.city or ""} · {rooms}</div>'
        f'<div style="margin-top:4px"><b>{price}</b> · score {r.score:.2f}</div>'
        f'<div style="color:#666;margin-top:4px;font-size:12px">{r.reason}</div>'
        f'</div>'
    )

def show_map(result: SearchResult, *, highlight_top: int = 1):
    pts = [r for r in result.ranked if r.listing.latitude is not None and r.listing.longitude is not None]
    if not pts:
        print('(no coordinates to plot)'); return None
    lat0 = sum(p.listing.latitude for p in pts) / len(pts)
    lon0 = sum(p.listing.longitude for p in pts) / len(pts)
    m = folium.Map(location=[lat0, lon0], zoom_start=12, tiles='CartoDB positron')
    for i, r in enumerate(pts):
        style = PIN_CSS_TOP if i < highlight_top else PIN_CSS
        price = r.listing.price_chf or 0
        html = f'<div style="{style}">CHF {price:,.0f}</div>'
        folium.Marker(
            [r.listing.latitude, r.listing.longitude],
            icon=DivIcon(icon_size=(90, 24), icon_anchor=(45, 12), html=html),
            popup=folium.Popup(_popup_html(r, i + 1), max_width=260),
            tooltip=f'#{i + 1} · {r.listing.title}',
        ).add_to(m)
    if len(pts) > 1:
        m.fit_bounds(
            [[min(p.listing.latitude for p in pts), min(p.listing.longitude for p in pts)],
             [max(p.listing.latitude for p in pts), max(p.listing.longitude for p in pts)]],
            padding=(30, 30),
        )
    return m

## 6. Debug dashboard

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

def _ranked_df(ranked: list[RankedListingResult]) -> pd.DataFrame:
    return pd.DataFrame([
        {'listing_id': r.listing_id, 'title': r.listing.title, 'city': r.listing.city,
         'price_chf': r.listing.price_chf, 'rooms': r.listing.rooms,
         'score': r.score, 'reason': r.reason}
        for r in ranked
    ])

def _candidates_df(candidates: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(candidates)[['listing_id', 'title', 'city', 'price', 'rooms']] if candidates else pd.DataFrame()

def dashboard(result: SearchResult) -> None:
    r = result
    display(Markdown(f'# 🔍 Query\n> {r.query}'))

    display(Markdown('## ⏱ Stage trace'))
    display(pd.DataFrame([{'stage': t.name, 'ms': round(t.ms, 2), **t.info} for t in r.traces]))

    display(Markdown('## 🧩 Extracted hard filters'))
    hf_dict = {k: v for k, v in r.hard_filters.__dict__.items() if v not in (None, [], 25, 0)}
    display(Markdown(f'`{hf_dict or "(none)"}`'))

    display(Markdown('## 🧩 Extracted soft facts'))
    display(Markdown(f'`{r.soft_facts}`'))

    display(Markdown(f'## 🧮 Candidates after hard filter — {len(r.candidates)} / {len(PSEUDO_POOL)}'))
    if r.candidates:
        display(_candidates_df(r.candidates))
    else:
        display(Markdown('_No listings survived the hard filter._'))

    display(Markdown(f'## 🧹 After soft filter — {len(r.soft_filtered)}'))

    display(Markdown(f'## 🏆 Ranked results — {len(r.ranked)}'))
    if r.ranked:
        display(_ranked_df(r.ranked))
    else:
        display(Markdown('_Nothing to rank._'))

def show_results(result: SearchResult) -> None:
    """Lean output: ranked table + map (no debug)."""
    display(Markdown(f'### Query\n> {result.query}'))
    if result.ranked:
        display(_ranked_df(result.ranked))
        m = show_map(result)
        if m is not None: display(m)
    else:
        display(Markdown('_No listings matched._'))

## 7. Demo — lean output with map

In [ ]:
show_results(search('3.5-room bright apartment in Zurich under 2800'))

## 8. Debug mode — full dashboard

In [ ]:
dashboard(search('3.5-room bright apartment in Zurich under 2800'))

In [ ]:
dashboard(search('modern quiet studio in Geneva'))

In [ ]:
dashboard(search('family friendly flat in Winterthur'))